# Module 2: From Clinical Question to Working Cohort
## ESICM LIVES26 — Datathon Educational Series

---

> **Prerequisites:** Complete Module 1 first. This notebook assumes you can connect to BigQuery, find concept IDs, and understand the `QUALIFY` pattern.
>
> **What this module covers:** The complete journey from a clinical hypothesis to a validated, analysis-ready cohort — using the actual pipeline built during the 2026 ESICM INDICATE datathon.

> **⚠ Datathon context:** This documents what we built and *why we made each decision*. Your datathon will present different problems requiring different solutions. The reasoning matters more than the specific SQL.

---

### Learning Objectives

By the end of this module, you will be able to:

1. **Translate** a clinical hypothesis into formal eligibility criteria with explicit inclusion/exclusion logic
2. **Construct** a reproducible cohort pipeline — from patient selection through feature extraction at two clinical landmarks to outcome classification
3. **Implement** fluid balance calculations using curated concept IDs with documented inclusion/exclusion decisions
4. **Validate** each pipeline step using distribution checks and clinical plausibility tests

> *Bloom's taxonomy level: Application — applying foundational skills from Module 1 to build a complete analytic pipeline.*

### The Journey

```
Clinical Question
      ↓
Who are our patients?        (Section 2)
      ↓
When did the key events happen?  (Section 3)
      ↓
What do we measure, and when?    (Section 4)
      ↓
Fluid balance — the core feature (Section 5)
      ↓
Outcomes — what actually happened (Section 6)
```

At every step: **🩺 Clinical Thinking → 💾 Data Translation → ✅ Sanity Check**

---
## Setup

Same configuration as Module 1. If you have already authenticated, skip the `auth.authenticate_user()` call.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
PROJECT_ID         = 'your-datathon-project'     # Your BigQuery project
DATASET_PROJECT_ID = 'amsterdamumcdb'            # AmsterdamUMCdb — do not change
DATASET_ID         = 'van_gogh_2026_datathon_update'  # Note: _update version used in final pipeline
LOCATION           = 'eu'

# Sandbox — where you build your working tables
SANDBOX_PROJECT    = 'your-datathon-project'     # Usually same as PROJECT_ID
SANDBOX_DATASET    = 'sandbox'                   # Your writable dataset
COHORT_TABLE       = f'{SANDBOX_PROJECT}.{SANDBOX_DATASET}.patient_weaning'

import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID

job_config = bigquery.job.QueryJobConfig()
def_config  = bigquery.job.QueryJobConfig(
    default_dataset=f'{DATASET_PROJECT_ID}.{DATASET_ID}'
)
client = bigquery.Client(
    project=PROJECT_ID, location=LOCATION,
    default_query_job_config=def_config
)
print(f'✓ Connected | Cohort table: {COHORT_TABLE}')

---
## Section 1: The Clinical Question

### 🩺 Clinical Thinking

You're an intensivist. Two patients, same morning:

| | Patient A | Patient B |
|---|---|---|
| Cumulative fluid balance | +2,400 mL | +2,400 mL |
| Trend over last 12h | Diuresing ↓ | Still accumulating ↑ |
| Your clinical plan | Attempt extubation | Wait another day |

**Same number. Opposite clinical decisions.**

This is the problem with snapshot-based assessment. The number alone doesn't tell you whether the patient is heading towards or away from a state of readiness.

### The Hypothesis

> **Dynamic fluid balance trajectories — the slope and direction of change — predict successful extubation better than any single measurement.**

If true, this has direct clinical implications:
- A patient diuresing towards neutral balance may be extubatable *before* they reach a target balance
- A patient accumulating despite diuretics may need intervention *before* a weaning attempt
- Trajectory is actionable; a snapshot is just a number

### The Gap Between Gestalt and Data

Experienced intensivists already do this — they look at the trend. But they do it intuitively, from memory of yesterday's numbers. What we're trying to do is **make the trajectory explicit, measurable, and testable at scale** across thousands of patients.

That requires building the right cohort — and that starts with a clear definition of who we're studying and when.

In [ ]:
# Illustrating the hypothesis: same endpoint, different trajectories
hours = list(range(0, 73, 6))

# Patient A: diuresing — fluid balance improving
patient_a = [3200, 3100, 2900, 2700, 2500, 2400, 2300, 2200, 2100, 2000, 1900, 1800, 1700]
# Patient B: still accumulating — same endpoint, different path
patient_b = [800, 1100, 1400, 1600, 1800, 2000, 2100, 2200, 2300, 2350, 2400, 2400, 2400]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, patient_a, 'g-o', linewidth=2, markersize=5, label='Patient A — diuresing ↓')
ax.plot(hours, patient_b, 'r-o', linewidth=2, markersize=5, label='Patient B — accumulating ↑')
ax.axhline(y=2400, color='gray', linestyle='--', alpha=0.5, label='Shared endpoint: +2,400 mL')
ax.axvline(x=72, color='black', linestyle=':', alpha=0.7)
ax.annotate('Weaning attempt', xy=(72, 2400), xytext=(60, 2800),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)
ax.set_xlabel('Hours before weaning attempt', fontsize=12)
ax.set_ylabel('Cumulative fluid balance (mL)', fontsize=12)
ax.set_title('Same number at weaning — very different trajectories', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('This is what we are trying to capture in data.')
print('The slope of the line — not the endpoint — is the signal.')

---
## Section 2: Who Are Our Patients?

### 🩺 Clinical Thinking

Before writing a single query, write your inclusion and exclusion criteria as if you were drafting a study protocol. Each criterion should have a clinical reason — **not** a data reason. The data reason comes second.

| Criterion | Clinical Reason |
|-----------|----------------|
| ✅ Invasive mechanical ventilation | NIV and CPAP are fundamentally different weaning problems — different physiology, different failure modes |
| ✅ IMV duration ≥ 24h | Brief post-operative ventilation (a few hours) is not a weaning scenario — it's planned extubation after anaesthesia |
| ❌ Neurological conditions | TBI, CVA, SAH patients fail extubation for neurological reasons (airway protection, GCS), not pulmonary ones — a different clinical question entirely |
| ❌ 2002 admission year | Data quality: fluid balance data is sparse for this cohort — results would be unreliable |

> **Pearl for future researchers:** Your exclusion criteria define your clinical question as much as your inclusion criteria. Excluding neuro patients is a statement that you're studying *pulmonary* weaning failure.

### 💾 Data Translation: Cohort Entry from Visit Occurrence

The starting point is `visit_occurrence` — ICU admissions. Two important data quirks discovered during the datathon:

1. **2002 is excluded** — fluid balance data was too sparse that year to produce reliable trajectories
2. **`DISTINCT` is required** — there are duplicate rows in `visit_occurrence`; without it you get inflated patient counts

In `visit_occurrence`, admission years are represented as timestamps starting on January 1st of that year (a consequence of date anonymisation). So filtering by year means filtering by `visit_start_datetime`.

In [ ]:
# Step 1: Verify what years are available and how many patients per year
# Always check before filtering — confirm your assumptions

query = f"""
SELECT
  EXTRACT(YEAR FROM visit_start_datetime) AS year,
  COUNT(DISTINCT person_id)              AS unique_patients,
  COUNT(*)                               AS total_rows,
  COUNT(*) - COUNT(DISTINCT person_id)   AS duplicate_rows
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.visit_occurrence`
GROUP BY year
ORDER BY year
"""

year_dist = client.query(query, job_config=job_config).to_dataframe()
print('Admissions by year:')
print(year_dist.to_string(index=False))
print('\n⚠ Note duplicate_rows > 0 — DISTINCT is required when building your cohort')
print('⚠ 2002 will be excluded — check fluid data availability there if you plan to include it')

In [ ]:
# Step 2: Create the base cohort table
# Include 2010 and 2017 only; use DISTINCT to handle duplicates

query_create = f"""
CREATE OR REPLACE TABLE `{COHORT_TABLE}` AS

-- DISTINCT because visit_occurrence contains duplicate rows
SELECT DISTINCT
  person_id,
  visit_start_datetime AS admission_start,
  visit_end_datetime   AS admission_end
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.visit_occurrence`
WHERE
  -- 2002 excluded: fluid data too sparse for reliable trajectories
  visit_start_datetime IN ('2017-01-01', '2010-01-01')
AND visit_end_datetime IS NOT NULL
"""

client.query(query_create, job_config=job_config).result()

# Verify
n = client.query(
    f'SELECT COUNT(*) AS n, COUNT(DISTINCT person_id) AS unique_pts FROM `{COHORT_TABLE}`',
    job_config=job_config
).to_dataframe()
print(f'✓ Base cohort created')
print(n.to_string(index=False))

### 🩺 Clinical Thinking: Why Exclude Neurological Patients?

This is one of the most important decisions in the cohort design — and one that is easy to miss if you're only thinking about data.

Patients with TBI, CVA, SAH, or intracranial haemorrhage fail extubation because they cannot protect their airway or maintain respiratory drive — not because of pulmonary pathology. Including them would:

- Dilute the fluid-trajectory signal (their failure is neurological, not fluid-related)
- Create a bimodal outcome distribution that obscures the effect you're trying to measure
- Produce a model that doesn't generalise to the clinical question

This is about **clinical validity**.

### 💾 Data Translation: Neuro Exclusion via `condition_occurrence`

Neurological conditions are identified from `condition_source_value` using clinical free-text patterns — both English and Dutch (AmsterdamUMCdb is a Dutch dataset).

In [ ]:
# Flag patients with neurological conditions during their ICU admission
# These patients will be excluded from cohort_criteria = 1 (but kept in the table for transparency)

query_neuro = f"""
UPDATE `{COHORT_TABLE}` pw
SET admission_has_neuro = 1
FROM (
  SELECT DISTINCT pw.person_id
  FROM `{COHORT_TABLE}` pw
  JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.condition_occurrence` co
    ON co.person_id = pw.person_id
  WHERE co.condition_start_datetime BETWEEN pw.admission_start AND pw.admission_end
  AND (
    -- Head trauma (English + Dutch)
    co.condition_source_value LIKE '%Head trauma%'
    OR co.condition_source_value LIKE '%Hoofdtrauma%'
    OR co.condition_source_value LIKE '%Head/multiple trauma%'
    -- Stroke / CVA
    OR co.condition_source_value LIKE '%CVA, cerebrovascular accident/stroke%'
    OR co.condition_source_value LIKE '%Subarachnoid hemorrhage%'
    OR co.condition_source_value LIKE '%S.A.B.%'
    -- Intracranial haemorrhage
    OR co.condition_source_value LIKE '%Intracerebral subdural or subarachnoid haemorrhage%'
    OR co.condition_source_value LIKE '%Hemorrhage/hematoma, intracranial%'
    OR co.condition_source_value LIKE '%Intracerebraal haematoom%'
    -- Subdural / epidural haematoma
    OR co.condition_source_value LIKE '%Hematoma, subdural%'
    OR co.condition_source_value LIKE '%Subduraal haematoom%'
  )
) flagged
WHERE pw.person_id = flagged.person_id
"""

client.query(query_neuro, job_config=job_config).result()

# ✅ Sanity check: how many flagged?
neuro_check = client.query(
    f"""SELECT
      COUNTIF(admission_has_neuro = 1) AS neuro_excluded,
      COUNTIF(admission_has_neuro IS NULL) AS non_neuro,
      ROUND(COUNTIF(admission_has_neuro = 1) * 100.0 / COUNT(*), 1) AS pct_excluded
    FROM `{COHORT_TABLE}`""",
    job_config=job_config
).to_dataframe()
print('✅ Neuro exclusion sanity check:')
print(neuro_check.to_string(index=False))
print('\nExpect ~5-15% of ICU patients to have neurological conditions')

### 🩺 Clinical Thinking: Identifying Invasive Mechanical Ventilation

As covered in Module 1 — you cannot rely on `procedure_occurrence` for this. You need to infer IMV from measurements.

The final approach used two concept IDs:
- `3022875` — PEEP setting (Ventilator)
- `36303816` — a second ventilator parameter (validate this in your instance using the Module 1 process)

**Why not use TV set as in Module 1?** During the datathon, this two-parameter combination was found to be more reliable for identifying the *start* of IMV. TV set (Module 1) is better for confirming *invasive* vs NIV. Both approaches are valid — the choice depends on your specific question.

**Gap tolerance: 6 hours, not 24.**  
A 24h gap in ventilator measurements almost always means a new episode. But a 6h gap might just mean a patient was briefly moved off the unit (radiology, OR) and still on the same IMV episode. The `device_exposure` table (made available late in the datathon) was used to validate this — gaps under 6h were confirmed to represent continuous episodes in most cases.

### 💾 Data Translation: IMV Start and Episode Boundaries

In [ ]:
# Identify first IMV measurement within the ICU admission
# Uses PEEP Set (3022875) + a second ventilator parameter (validate concept 36303816 in your instance)

IMV_CONCEPT_IDS = [36303816, 3022875]  # ← validate these in your instance using Module 1 process
ids_str = ','.join(str(i) for i in IMV_CONCEPT_IDS)

query_imv_start = f"""
UPDATE `{COHORT_TABLE}` p
SET
  first_imv_start = ilv.first_ts
FROM (
  SELECT
    m.person_id,
    MIN(m.measurement_datetime) AS first_ts
  FROM `{COHORT_TABLE}` p
  JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m ON m.person_id = p.person_id
  WHERE
    m.measurement_concept_id IN ({ids_str})
  AND m.measurement_datetime >= p.admission_start
  GROUP BY m.person_id
) ilv
WHERE p.person_id = ilv.person_id
"""

client.query(query_imv_start, job_config=job_config).result()
print('✓ First IMV start timestamp set')

In [ ]:
# Find end of first IMV episode — using 6h gap tolerance
# Logic: find the LAST measurement before a gap of ≥6h (or the absolute last measurement)
# A gap ≥6h = start of a new episode (or end of ventilation)

query_imv_end = f"""
UPDATE `{COHORT_TABLE}` p
SET
  first_imv_end = ilv.segment_end_ts,
  second_imv_start = ilv.next_segment_ts
FROM (
  WITH cte_ts AS (
    SELECT
      m.person_id,
      m.measurement_datetime,
      LEAD(m.measurement_datetime) OVER (
        PARTITION BY m.person_id ORDER BY m.measurement_datetime
      ) AS next_ts
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
    JOIN `{COHORT_TABLE}` p ON p.person_id = m.person_id
    WHERE
      m.measurement_concept_id IN ({ids_str})
    AND m.measurement_datetime >= p.first_imv_start
  )
  SELECT
    person_id,
    measurement_datetime AS segment_end_ts,
    next_ts              AS next_segment_ts,
    ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY measurement_datetime) AS row_num
  FROM cte_ts
  WHERE next_ts IS NULL
     OR TIMESTAMP_DIFF(next_ts, measurement_datetime, HOUR) >= 6  -- 6h gap = new episode
) ilv
WHERE p.person_id = ilv.person_id
AND   ilv.row_num = 1
"""

client.query(query_imv_end, job_config=job_config).result()

# ✅ Sanity check: IMV duration distribution
imv_check = client.query(f"""
  SELECT
    COUNT(*)                                                      AS total_patients,
    COUNTIF(first_imv_start IS NOT NULL)                          AS has_imv,
    ROUND(AVG(TIMESTAMP_DIFF(first_imv_end, first_imv_start, HOUR)), 1) AS mean_imv_hours,
    MIN(TIMESTAMP_DIFF(first_imv_end, first_imv_start, HOUR))     AS min_imv_hours,
    MAX(TIMESTAMP_DIFF(first_imv_end, first_imv_start, HOUR))     AS max_imv_hours
  FROM `{COHORT_TABLE}`
  WHERE first_imv_end IS NOT NULL""", job_config=job_config).to_dataframe()
print('✅ IMV episode sanity check:')
print(imv_check.to_string(index=False))
print('\nExpect mean ~100-200h for an ICU-level IMV cohort')

### 💾 Finalising Cohort Inclusion

A patient enters the final analysis cohort (`cohort_criteria = 1`) if ALL of the following are true:

| Criterion | Clinical Meaning |
|-----------|------------------|
| `first_weaning IS NOT NULL` | They actually reached a weaning event (Section 3 will define this) |
| `first_imv_duration ≥ 24h` | Not a brief post-op ventilation — a genuine ICU ventilation episode |
| `admission_has_neuro = 0` | No neurological conditions that confound weaning assessment |

---
## Section 3: Two Timestamps — The Key Clinical Insight

### 🩺 Clinical Thinking: Readiness ≠ Action

Imagine you're doing a 8am ward round. The patient meets all your weaning criteria:
- FiO₂ ≤ 40%, PEEP ≤ 8 cmH₂O
- Off vasopressors overnight
- Awake, following commands, coughing
- No NMRA or sedation in 6 hours

But you don't extubate until 2pm. What happened in those 6 hours?
- Physiotherapy assessment
- Waiting for senior review
- Spontaneous breathing trial
- Clinical conservatism — "let's see how they do on lower support"

**The gap between *eligible* and *extubated* is clinically meaningful.** A patient who deteriorates in that window is different from one who continues to improve. That gap — and what happens during it — is exactly what our trajectory features capture.

### The Two Timestamps

```
IMV start ──────────────────── Eligibility ────────────── Weaning (IMV end)
          ←── ventilated ────→ ←──── gap ────────────────→
                              ↑                           ↑
                    First moment criteria met      IMV actually stopped
                    (feature snapshot #1)          (feature snapshot #2)
                                                   slope = (snapshot2 - snapshot1) / hours
```

The **slope** between these two snapshots is the operationalised trajectory hypothesis.

### 💾 Eligibility Timestamp: When Were They First Truly Ready?

Eligibility criteria — all must be met simultaneously:

| Criterion | Threshold | Clinical Rationale |
|-----------|-----------|--------------------|
| FiO₂ | ≤ 50% (or ≤ 0.5 fraction) | Standard low-support definition |
| PEEP | ≤ 10 cmH₂O | Standard low-support definition |
| Noradrenaline | None active | Vasopressor dependence = haemodynamically unstable |
| Propofol | None active (3h washout) | Sedation impairs respiratory drive and airway reflexes |
| NMRA | None active | Neuromuscular blockade = cannot breathe spontaneously |
| Timing | ≥ 6h after admission start | Exclude drugs given before ICU transfer |

> **Why these drugs specifically?** Because if a patient is on noradrenaline, propofol, or a neuromuscular relaxant, no competent clinician would be considering extubation — regardless of what the ventilator settings show. Including such patients as 'eligible' would be clinically nonsensical.

In [ ]:
# Drug concept IDs — validate these in your instance
NORAD_CONCEPT_ID  = 1321341   # Noradrenaline / norepinephrine
PROPOFOL_CONCEPT_ID = 753626  # Propofol
NMRA_CONCEPT_ID   = 19003953  # Neuromuscular relaxant

# Find first eligibility timestamp:
# First moment FiO2 ≤50% AND PEEP ≤10 are both met
# WITH none of the three exclusion drugs active at that time

query_eligible = f"""
UPDATE `{COHORT_TABLE}` w
SET
  eligible_start_time = ilv.start_time,
  eligible_start_fio2 = ilv.fio2_value,
  eligible_start_peep = ilv.peep_value
FROM (
  WITH cohort AS (
    SELECT * FROM `{COHORT_TABLE}` WHERE first_imv_end IS NOT NULL
  )
  SELECT
    m_fio2.person_id,
    MIN(m_fio2.measurement_datetime) AS start_time,
    CAST(ROUND(APPROX_QUANTILES(m_fio2.value_as_number, 2)[OFFSET(1)], 2) AS NUMERIC) AS fio2_value,
    CAST(ROUND(APPROX_QUANTILES(m_peep.value_as_number, 2)[OFFSET(1)], 2) AS NUMERIC) AS peep_value
  FROM cohort p
  JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m_fio2 ON m_fio2.person_id = p.person_id
  JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m_peep ON m_peep.person_id = p.person_id
  WHERE
    -- FiO2 criteria: ≤50% (handle both fraction and percentage encoding)
    m_fio2.measurement_concept_id = 2000000204
    AND (m_fio2.value_as_number BETWEEN 1 AND 50 OR m_fio2.value_as_number BETWEEN 0 AND 0.5)
    AND m_fio2.unit_source_value != '%'
    AND m_fio2.measurement_datetime BETWEEN p.first_imv_start AND p.first_imv_end
    -- PEEP criteria: ≤10 cmH2O
    AND m_peep.measurement_concept_id = 3022875
    AND m_peep.value_as_number <= 10
    AND m_peep.measurement_datetime BETWEEN p.first_imv_start AND p.first_imv_end
    -- Both measured within 30 minutes of each other
    AND m_fio2.measurement_datetime
          BETWEEN TIMESTAMP_SUB(m_peep.measurement_datetime, INTERVAL 30 MINUTE)
              AND TIMESTAMP_ADD(m_peep.measurement_datetime, INTERVAL 30 MINUTE)
    -- ≥6h after admission (exclude pre-transfer drugs)
    AND m_fio2.measurement_datetime >= TIMESTAMP_ADD(p.admission_start, INTERVAL 6 HOUR)
    -- Exclusion: no noradrenaline active
    AND NOT EXISTS (
      SELECT 1 FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.drug_exposure` de
      WHERE de.drug_concept_id = {NORAD_CONCEPT_ID}
      AND de.person_id = p.person_id
      AND m_fio2.measurement_datetime
            BETWEEN TIMESTAMP_SUB(de.drug_exposure_start_datetime, INTERVAL 6 HOUR)
                AND TIMESTAMP_ADD(de.drug_exposure_end_datetime, INTERVAL 3 HOUR)
    )
    -- Exclusion: no propofol active
    AND NOT EXISTS (
      SELECT 1 FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.drug_exposure` de
      WHERE de.drug_concept_id = {PROPOFOL_CONCEPT_ID}
      AND de.person_id = p.person_id
      AND m_fio2.measurement_datetime
            BETWEEN TIMESTAMP_SUB(de.drug_exposure_start_datetime, INTERVAL 6 HOUR)
                AND TIMESTAMP_ADD(de.drug_exposure_end_datetime, INTERVAL 3 HOUR)
    )
    -- Exclusion: no neuromuscular relaxant active
    AND NOT EXISTS (
      SELECT 1 FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.drug_exposure` de
      WHERE de.drug_concept_id = {NMRA_CONCEPT_ID}
      AND de.person_id = p.person_id
      AND m_fio2.measurement_datetime
            BETWEEN TIMESTAMP_SUB(de.drug_exposure_start_datetime, INTERVAL 6 HOUR)
                AND TIMESTAMP_ADD(de.drug_exposure_end_datetime, INTERVAL 3 HOUR)
    )
  GROUP BY m_fio2.person_id
) ilv
WHERE w.person_id = ilv.person_id
"""

client.query(query_eligible, job_config=job_config).result()

# ✅ Sanity check
elig_check = client.query(f"""
  SELECT
    COUNTIF(eligible_start_time IS NOT NULL) AS has_eligibility,
    COUNTIF(eligible_start_time IS NULL AND first_imv_end IS NOT NULL) AS imv_but_no_eligibility,
    ROUND(AVG(TIMESTAMP_DIFF(first_imv_end, eligible_start_time, HOUR)), 1) AS mean_gap_hours
  FROM `{COHORT_TABLE}`""", job_config=job_config).to_dataframe()
print('✅ Eligibility sanity check:')
print(elig_check.to_string(index=False))
print('\nMean gap >0 confirms readiness ≠ weaning — the clinical gap exists in the data')

### 💾 The Weaning Timestamp

**Weaning = end of first IMV episode.**

We don't try to find a specific 'extubation' event — that doesn't exist cleanly in the data. Instead, we define weaning as the moment IMV measurably stopped: the end of the first continuous IMV episode detected in Section 2.

This is a pragmatic approximation. It will occasionally capture:
- Patients who were briefly taken off the ventilator and immediately reintubated (captured as failure in outcomes)
- Patients who died on the ventilator (handled by the death-on-ventilator logic in Section 6)

Both are edge cases and both have clinical meaning — they're not noise.

In [ ]:
# Set weaning timestamp = end of first IMV episode
# Only for patients who are alive at weaning and were eligible

query_weaning = f"""
UPDATE `{COHORT_TABLE}`
SET first_weaning = first_imv_end
WHERE
  first_imv_end IS NOT NULL
AND first_imv_end_eligible = 1           -- Must have reached eligibility criteria
AND (died_ts IS NULL
     OR TIMESTAMP_DIFF(died_ts, first_imv_end, HOUR) >= 12)  -- Not dying at the moment of extubation
AND admission_has_neuro = 0              -- Not a neurological patient
"""

client.query(query_weaning, job_config=job_config).result()

# Calculate duration: eligibility to weaning
query_duration = f"""
UPDATE `{COHORT_TABLE}`
SET duration_eligible_to_weaning = TIMESTAMP_DIFF(first_imv_end, eligible_start_time, HOUR)
WHERE first_imv_end IS NOT NULL AND eligible_start_time IS NOT NULL
"""
client.query(query_duration, job_config=job_config).result()

# ✅ Sanity check: the gap distribution
gap_check = client.query(f"""
  SELECT
    COUNT(*)                                                AS patients_with_weaning,
    ROUND(AVG(duration_eligible_to_weaning), 1)             AS mean_gap_h,
    MIN(duration_eligible_to_weaning)                       AS min_gap_h,
    MAX(duration_eligible_to_weaning)                       AS max_gap_h,
    COUNTIF(duration_eligible_to_weaning < 0)               AS impossible_negative_gaps
  FROM `{COHORT_TABLE}`
  WHERE first_weaning IS NOT NULL""", job_config=job_config).to_dataframe()
print('✅ Eligibility-to-weaning gap:')
print(gap_check.to_string(index=False))
print('\nNegative gaps = data problem (eligibility detected after IMV end) — investigate if >0')
print('Large mean gap (>24h) suggests eligibility is being detected too early — review criteria')

---
## Section 4: What Do We Measure, and When?

### 🩺 Clinical Thinking: Two Snapshots, One Trajectory

We measure each clinical variable at **both** timestamps — eligibility and weaning. This gives us:

1. **The state at eligibility** — what the patient looked like when they first met criteria
2. **The state at weaning** — what they looked like when extubated
3. **The slope** — the trajectory between the two

Why not just measure at weaning? Because two patients with identical values at weaning may have gotten there very differently. The one who was already at that value 12 hours ago is different from the one who just got there.

### The Measurement Windows

| Variable type | Window | Clinical rationale |
|---------------|--------|--------------------|
| Ventilator settings (FiO₂, PEEP) | ±2h | Change frequently — want the closest reading |
| Haemodynamics (SBP, DBP) | ±4h | Measured continuously but averaged over longer windows |
| SpO₂ | ±12h | Continuous, but broader window tolerates gaps |
| Blood tests (Hb, WBC, CRP, lactate, etc.) | ±12h | Labs are not measured hourly — ±12h captures the nearest result |

### 💾 The QUALIFY Pattern

For each variable, we want the **single closest measurement** to the landmark time. `QUALIFY ROW_NUMBER()` does this efficiently in BigQuery — it's the right tool for 'pick the nearest value' problems.

In [ ]:
# The QUALIFY pattern — explained
# This template works for any variable at any landmark time
# Replace: CONCEPT_ID, window_hours_before, window_hours_after, value filter, landmark field

CONCEPT_ID_EXAMPLE = 2000000204  # FiO2 — replace with your variable
LANDMARK = 'first_weaning'       # or 'eligible_start_time'

query_template = f"""
-- Template: extract closest measurement to a landmark time
SELECT
  m.person_id,
  m.measurement_datetime,
  m.value_as_number,
  -- How far from the landmark?
  TIMESTAMP_DIFF(pw.{LANDMARK}, m.measurement_datetime, MINUTE) AS minutes_from_landmark
FROM `{COHORT_TABLE}` pw
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m ON m.person_id = pw.person_id
WHERE
  pw.cohort_criteria = 1
  AND m.measurement_concept_id = {CONCEPT_ID_EXAMPLE}
  -- Define the window: 2h before to landmark time
  AND m.measurement_datetime BETWEEN
    DATETIME_SUB(pw.{LANDMARK}, INTERVAL 2 HOUR)
    AND pw.{LANDMARK}
  -- Value plausibility filter
  AND m.value_as_number BETWEEN 0.1 AND 100
  AND m.unit_source_value != '%'
QUALIFY
  -- This is the key: pick only the row closest to the landmark, per patient
  ROW_NUMBER() OVER (
    PARTITION BY m.person_id
    ORDER BY ABS(TIMESTAMP_DIFF(pw.{LANDMARK}, m.measurement_datetime, HOUR))
  ) = 1
LIMIT 10
"""

demo = client.query(query_template, job_config=job_config).to_dataframe()
print('QUALIFY pattern demonstration (10 rows):')
print(demo.to_string(index=False))
print('\nEach patient_id appears exactly once — the closest measurement wins.')

### ⚠ Unit Handling Pitfalls

These were discovered during the datathon — they are not in the documentation:

| Variable | Pitfall | Fix |
|----------|---------|-----|
| **FiO₂** | Stored as both fraction (0.21–1.0) and percentage (21–100) in different records | Detect and multiply: `CASE WHEN value BETWEEN 0.1 AND 1 THEN value * 100 ELSE value END` |
| **FiO₂** | Records with `unit_source_value = '%'` are all 100 — a data entry artefact | Exclude: `AND unit_source_value != '%'` |
| **Hb** | Must filter to `unit_source_value LIKE 'mmol/l'` — other units present | Without this filter, values from different units are mixed and the distribution is nonsensical |
| **HCT** | Stored as fraction (0.30–0.50), not percentage | Filter: `BETWEEN 0.1 AND 1` — do not convert |
| **SpO₂** | Same fraction/percentage split as FiO₂ | Same CASE fix |

In [ ]:
# Feature extraction at WEANING timestamp
# Showing FiO2 and PEEP as representative examples of the pattern
# The same UPDATE pattern is repeated for all 14 features

# FiO2 at weaning — with unit handling
query_fio2_weaning = f"""
UPDATE `{COHORT_TABLE}` pw
SET
  fio2_at_weaning          = ROUND(CAST(ilv.measurement_value AS NUMERIC), 4),
  fio2_at_weaning_datetime = ilv.measurement_datetime
FROM (
  SELECT
    m.person_id,
    m.measurement_datetime,
    -- Unit handling: fraction → percentage
    CASE WHEN m.value_as_number BETWEEN 0.1 AND 1
         THEN m.value_as_number * 100
         ELSE m.value_as_number
    END AS measurement_value
  FROM `{COHORT_TABLE}` pw
  JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m ON m.person_id = pw.person_id
  WHERE
    pw.cohort_criteria = 1
  AND m.measurement_concept_id = 2000000204  -- FiO2 setting
  AND m.measurement_datetime BETWEEN
    DATETIME_SUB(pw.first_weaning, INTERVAL 2 HOUR) AND pw.first_weaning
  AND m.value_as_number BETWEEN 0.1 AND 100
  AND m.unit_source_value != '%'             -- Exclude artefactual 100% records
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY m.person_id
    ORDER BY ABS(TIMESTAMP_DIFF(pw.first_weaning, m.measurement_datetime, HOUR))
  ) = 1
) ilv
WHERE ilv.person_id = pw.person_id
"""

client.query(query_fio2_weaning, job_config=job_config).result()

# ✅ Sanity check FiO2
fio2_check = client.query(f"""
  SELECT
    COUNT(fio2_at_weaning) AS has_fio2,
    ROUND(AVG(fio2_at_weaning), 1) AS mean_fio2,
    MIN(fio2_at_weaning) AS min_fio2,
    MAX(fio2_at_weaning) AS max_fio2
  FROM `{COHORT_TABLE}` WHERE cohort_criteria = 1""",
  job_config=job_config).to_dataframe()
print('✅ FiO2 at weaning sanity check:')
print(fio2_check.to_string(index=False))
print('\nExpect: all values 21–100, mean ~35–45% at weaning (low support)')

### The Full Feature Set

The same `UPDATE ... QUALIFY ROW_NUMBER()` pattern is applied for all 14 features, at both eligibility and weaning timestamps (28 UPDATE queries total in the full pipeline).

| Feature | Concept ID(s) | Window | Unit note |
|---------|--------------|--------|-----------|
| FiO₂ | 2000000204 | ±2h pre | Fraction→% fix required |
| PEEP | 3022875 | ±2h pre | — |
| SBP (invasive) | 21490853 | ±4h | — |
| DBP (invasive) | 21490851 | ±12h | — |
| SpO₂ | 40762499, 3016502 | ±12h | Fraction→% fix required |
| Haemoglobin | 40762351, 21490721 | ±12h | mmol/L filter required |
| WBC | 3010813 | ±12h | — |
| Platelets | 3007461 | ±12h | — |
| Base excess | 3003396 | ±12h | — |
| CRP | 3020460 | ±12h | — |
| Creatinine | 3020564 | ±12h | — |
| Albumin | 3024561 | ±12h | — |
| Haematocrit | 3034976, 3009542 | ±12h | Fraction (0.1–1), do NOT convert |
| Lactate | 3047181, 3018405 | ±12h | — |

> **Apply the Module 1 process** to validate each concept ID in your datathon instance before building on it.

### 🩺 The Slope — Operationalising the Trajectory Hypothesis

This is the core innovation. The slope converts two static snapshots into a dynamic measure:

```
slope = (value at weaning − value at eligibility) / hours between
```

**Clinical interpretation:**
- FiO₂ slope < 0 → FiO₂ requirements *decreasing* → patient improving
- FiO₂ slope > 0 → FiO₂ requirements *increasing* → patient deteriorating between eligible and extubated
- Fluid balance slope > 0 → still accumulating
- Fluid balance slope < 0 → diuresing

**Why require a minimum 4-hour gap?** A slope calculated over 30 minutes of noise is meaningless. We require at least 4 hours between the two timestamps for the slope to represent a real clinical trend.

**This is the mathematical operationalisation of clinical gestalt** — the intensivist's intuition about direction of travel, encoded as a number.

In [ ]:
# Calculate slopes for all features
# slope = (value_at_weaning - value_at_eligible) / hours_between
# Only calculated when gap >= 4 hours (otherwise clinically meaningless)

query_slopes = f"""
UPDATE `{COHORT_TABLE}` p
SET
  fio2_slope = CASE
    WHEN TIMESTAMP_DIFF(fio2_at_weaning_datetime, fio2_at_eligible_datetime, HOUR) > 4
    THEN (fio2_at_weaning - fio2_at_eligible)
          / TIMESTAMP_DIFF(fio2_at_weaning_datetime, fio2_at_eligible_datetime, HOUR)
    ELSE NULL  -- Gap too short: slope would be noise
    END,
  peep_slope = CASE
    WHEN TIMESTAMP_DIFF(peep_at_weaning_datetime, peep_at_eligible_datetime, HOUR) > 4
    THEN (peep_at_weaning - peep_at_eligible)
          / TIMESTAMP_DIFF(peep_at_weaning_datetime, peep_at_eligible_datetime, HOUR)
    ELSE NULL
    END
  -- ... same pattern for all 14 features
WHERE cohort_criteria = 1
"""

client.query(query_slopes, job_config=job_config).result()

# ✅ Sanity check: slope distributions
slope_check = client.query(f"""
  SELECT
    ROUND(AVG(fio2_slope), 4)  AS mean_fio2_slope,
    ROUND(AVG(peep_slope), 4)  AS mean_peep_slope,
    COUNTIF(fio2_slope IS NULL) AS fio2_slope_missing,
    COUNT(*)                    AS total
  FROM `{COHORT_TABLE}`
  WHERE cohort_criteria = 1""", job_config=job_config).to_dataframe()
print('✅ Slope sanity check:')
print(slope_check.to_string(index=False))
print('\nNegative FiO2 slope = improving (expected for patients successfully weaned)')
print('Missing slopes = timestamps too close together (<4h gap)')

---
## Section 5: Fluid Balance — The Core Feature

### 🩺 Clinical Thinking

Fluid balance is the central variable in our hypothesis. But it's also the hardest to extract correctly.

Unlike ventilator settings (recorded every hour by the ventilator) or blood tests (ordered and resulted), fluid balance is the **sum of dozens of different measurement streams** — IV infusions, enteral feeds, urine output, drains, blood products, dialysis. Each has its own concept ID, its own measurement pattern, and its own data quality issues.

**Getting this right requires investigative work.** As shown in Module 1, there are no pre-calculated fluid balances in AmsterdamUMCdb — you build it from components.

### Three Time Windows

```
IMV start ────────────── Eligibility ────────────── Weaning
←── Period 1 ──────────→ ←── Period 2 ──────────────→
                         ←── 24h prior to weaning ───→
```

Why three windows? Because the trajectory during different phases of illness may carry different predictive value:
- Period 1 (IMV to eligible): the whole ventilation episode — cumulative burden
- Period 2 (eligible to weaning): what happened after the patient was deemed ready
- 24h prior: the most recent clinical window — what the bedside team would be looking at

### 💾 The Curated Concept ID Lists

This is the product of careful manual review during the datathon. **Every ID below was individually checked** for whether it represents an incremental measurement or a cumulative total. The distinction matters enormously for summing.

#### Intake — all confirmed incremental (safe to SUM)

| Concept ID | Description |
|------------|-------------|
| 3037253 | Fluid intake intravascular Measured |
| 3004654 | Transfusion volume |
| 3010494 | Fluid intake enteral tube Measured |
| 2000000085 | Fluid intake unspecified category |
| 3006552 | Fluid intake oral Estimated |

#### Output — USE vs DO NOT USE

| Concept ID | Description | Status |
|------------|-------------|--------|
| 3014315 | Urine output | ✅ USE |
| **3008828** | **Urine output 8-hour** | **❌ DO NOT USE — moving 8h window, will double-count** |
| 2000000063 | Ultrafiltration SET CVVH | ✅ USE |
| **2000000064** | **Ultrafiltration TOTAL CVVH** | **❌ DO NOT USE — cumulative running total** |
| 21493943 | Blood loss Measured | ✅ USE |
| 2000000090 | Fluid output drain unspecified | ✅ USE |
| 3026556 | Fluid output chest tube | ✅ USE (large numbers) |
| 3020433 | Fluid output miscellaneous | ✅ USE (large numbers) |
| 3006376 | Fluid output wound drain | ✅ USE |
| 21491183 | GI drain | ✅ USE (90% confident incremental) |
| 3011087 | Output stool | ✅ USE |
| 3008793 | Biliary drain | ✅ USE |
| 2000000364 | Ileostoma output | ✅ USE |
| 2000000370 | Colostoma output | ✅ USE |
| **2000000232** | **Tracheobronchial toilet** | **❌ DO NOT USE — typically ~10mL, negligible** |

> **Why this matters:** Using `3008828` (8h moving window) instead of `3014315` (raw urine output) will massively inflate your output totals because each hour's measurement includes the prior 8 hours. The result looks plausible but is completely wrong.

In [ ]:
# Intake concept IDs — all confirmed incremental
INTAKE_IDS = [
    3037253,    # Fluid intake intravascular
    3004654,    # Transfusion volume
    3010494,    # Fluid intake enteral tube
    2000000085, # Fluid intake unspecified
    3006552,    # Fluid intake oral
]

# Output concept IDs — carefully curated; exclusions commented out
OUTPUT_IDS = [
    3014315,    # Urine output (use this, NOT 3008828)
    # 3008828 — DO NOT USE: 8h moving window, double-counts
    2000000063, # Ultrafiltration SET CVVH (use this, NOT 2000000064)
    # 2000000064 — DO NOT USE: cumulative running total
    21493943,   # Blood loss
    2000000090, # Drain unspecified
    3026556,    # Chest tube
    3020433,    # Miscellaneous output
    3006376,    # Wound drain
    21491183,   # GI drain
    3011087,    # Stool
    3008793,    # Biliary drain
    2000000364, # Ileostoma
    2000000370, # Colostoma
    # 2000000232 — DO NOT USE: tracheobronchial toilet ~10mL negligible
]

intake_str = ','.join(str(i) for i in INTAKE_IDS)
output_str = ','.join(str(i) for i in OUTPUT_IDS)

# Calculate fluid intake: IMV start to eligibility
query_intake = f"""
UPDATE `{COHORT_TABLE}` w
SET fluid_intake_imv_to_eligible = ilv.value
FROM (
  SELECT
    m.person_id,
    CAST(ROUND(SUM(m.value_as_number), 2) AS NUMERIC) AS value
  FROM `{COHORT_TABLE}` p
  JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m ON m.person_id = p.person_id
  WHERE
    p.first_imv_end IS NOT NULL
  AND m.measurement_concept_id IN ({intake_str})
  AND m.measurement_datetime BETWEEN p.first_imv_start AND p.eligible_start_time
  GROUP BY m.person_id
) ilv
WHERE w.person_id = ilv.person_id
"""

client.query(query_intake, job_config=job_config).result()
print('✓ Fluid intake (IMV→eligible) calculated')

# ✅ Sanity check fluid balance
fluid_check = client.query(f"""
  SELECT
    ROUND(AVG(fluid_intake_imv_to_eligible), 0)  AS mean_intake_ml,
    MIN(fluid_intake_imv_to_eligible)             AS min_intake_ml,
    MAX(fluid_intake_imv_to_eligible)             AS max_intake_ml,
    COUNTIF(fluid_intake_imv_to_eligible < 0)     AS impossible_negative_intake
  FROM `{COHORT_TABLE}`
  WHERE cohort_criteria = 1""", job_config=job_config).to_dataframe()
print('\n✅ Fluid intake sanity check:')
print(fluid_check.to_string(index=False))
print('\nExpect: positive values, mean likely 10,000–30,000 mL over full IMV episode')
print('Negative intake = impossible — investigate immediately')

### 💾 Period 1: Output (IMV start → Eligibility)

Same concept IDs as above, same window. Output SUM is straightforward — all verified incremental.

In [ ]:
# Period 1: fluid output (IMV start → eligibility)
query_output_p1 = f"""
UPDATE `{COHORT_TABLE}` w
SET fluid_output_imv_to_eligible = ilv.value
FROM (
  SELECT
    m.person_id,
    CAST(ROUND(SUM(m.value_as_number), 2) AS NUMERIC) AS value
  FROM `{COHORT_TABLE}` p
  JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m ON m.person_id = p.person_id
  WHERE
    p.first_imv_end IS NOT NULL
  AND m.measurement_concept_id IN ({output_str})
  AND m.measurement_datetime BETWEEN p.first_imv_start AND p.eligible_start_time
  GROUP BY m.person_id
) ilv
WHERE w.person_id = ilv.person_id
"""
client.query(query_output_p1, job_config=job_config).result()
print('✓ Fluid output (IMV→eligible) calculated')

In [ ]:
# Period 2: intake and output (eligibility → weaning)
# This is the most clinically meaningful window — happens AFTER the patient was deemed ready

query_intake_p2 = f"""
UPDATE `{COHORT_TABLE}` w
SET fluid_intake_eligible_to_weaning = ilv.value
FROM (
  SELECT
    m.person_id,
    CAST(ROUND(SUM(m.value_as_number), 2) AS NUMERIC) AS value
  FROM `{COHORT_TABLE}` p
  JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m ON m.person_id = p.person_id
  WHERE
    p.first_imv_end IS NOT NULL
  AND m.measurement_concept_id IN ({intake_str})
  AND m.measurement_datetime BETWEEN p.eligible_start_time AND p.first_weaning
  GROUP BY m.person_id
) ilv
WHERE w.person_id = ilv.person_id
"""
client.query(query_intake_p2, job_config=job_config).result()

query_output_p2 = f"""
UPDATE `{COHORT_TABLE}` w
SET fluid_output_eligible_to_weaning = ilv.value
FROM (
  SELECT
    m.person_id,
    CAST(ROUND(SUM(m.value_as_number), 2) AS NUMERIC) AS value
  FROM `{COHORT_TABLE}` p
  JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m ON m.person_id = p.person_id
  WHERE
    p.first_imv_end IS NOT NULL
  AND m.measurement_concept_id IN ({output_str})
  AND m.measurement_datetime BETWEEN p.eligible_start_time AND p.first_weaning
  GROUP BY m.person_id
) ilv
WHERE w.person_id = ilv.person_id
"""
client.query(query_output_p2, job_config=job_config).result()
print('✓ Period 2 (eligible→weaning) intake and output calculated')

### 🩺 Why a 24h Window?

A bedside intensivist reviewing a patient in the morning doesn't typically think "what was the fluid balance over the entire 14-day ventilation?" They think "how has the patient been in the last day?"

The 24h-prior window captures the most **clinically salient** period — the window immediately before extubation that the treating team would actually be assessing. Including all three windows lets the model determine which period's trajectory carries the most predictive weight.

In [ ]:
# 24h prior to weaning: intake, output, balance
# Window: 24h before first_weaning to first_weaning

# Initialise to 0 first (ensures no NULLs for patients without measurements in this window)
client.query(f"""
  UPDATE `{COHORT_TABLE}`
  SET fluid_intake_24h_prior = 0, fluid_output_24h_prior = 0, fluid_balance_24h_prior = 0
  WHERE fluid_balance_24h_prior IS NULL AND first_imv_end IS NOT NULL
""", job_config=job_config).result()

query_intake_24h = f"""
UPDATE `{COHORT_TABLE}` w
SET fluid_intake_24h_prior = ilv.value
FROM (
  SELECT
    m.person_id,
    CAST(ROUND(SUM(m.value_as_number), 2) AS NUMERIC) AS value
  FROM `{COHORT_TABLE}` p
  JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m ON m.person_id = p.person_id
  WHERE
    p.first_imv_end IS NOT NULL
  AND m.measurement_concept_id IN ({intake_str})
  AND m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(p.first_weaning, INTERVAL 24 HOUR) AND TIMESTAMP(p.first_weaning)
  GROUP BY m.person_id
) ilv
WHERE w.person_id = ilv.person_id
"""
client.query(query_intake_24h, job_config=job_config).result()

query_output_24h = f"""
UPDATE `{COHORT_TABLE}` w
SET fluid_output_24h_prior = ilv.value
FROM (
  SELECT
    m.person_id,
    CAST(ROUND(SUM(m.value_as_number), 2) AS NUMERIC) AS value
  FROM `{COHORT_TABLE}` p
  JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m ON m.person_id = p.person_id
  WHERE
    p.first_imv_end IS NOT NULL
  AND m.measurement_concept_id IN ({output_str})
  AND m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(p.first_weaning, INTERVAL 24 HOUR) AND TIMESTAMP(p.first_weaning)
  GROUP BY m.person_id
) ilv
WHERE w.person_id = ilv.person_id
"""
client.query(query_output_24h, job_config=job_config).result()
print('✓ 24h-prior intake and output calculated')

In [ ]:
# Derived columns: fluid balance = intake - output for each period
# Also: duration_imv_to_eligible (needed for slopes)
client.query(f"""
  UPDATE `{COHORT_TABLE}`
  SET
    duration_imv_to_eligible          = TIMESTAMP_DIFF(eligible_start_time, first_imv_start, HOUR),
    fluid_balance_imv_to_eligible     = fluid_intake_imv_to_eligible - fluid_output_imv_to_eligible,
    fluid_balance_eligible_to_weaning = fluid_intake_eligible_to_weaning - fluid_output_eligible_to_weaning,
    fluid_balance_24h_prior           = fluid_intake_24h_prior - fluid_output_24h_prior
  WHERE first_imv_end IS NOT NULL
""", job_config=job_config).result()
print('✓ Fluid balance derived columns calculated')

### 🩺 Fluid Balance Slopes — The Hypothesis in Numbers

The slopes are the **core engineered features** of this study. For fluid balance:

```
fluid_balance_eligible_to_weaning_slope = balance_eligible_to_weaning / duration_eligible_to_weaning
```

A **negative slope** means the patient is diuresing — net fluid is leaving — positive change from a clinical perspective.  
A **positive slope** means continued accumulation.

This is the number our hypothesis predicts will be different between success and failure.

**Important:** We normalise by *duration*, not by a fixed window. This makes slopes from a 4h eligible-to-weaning period directly comparable to a 48h period — both express change *per hour*.

In [ ]:
# Fluid balance slopes: balance / duration (mL/hour)
# Three slope sets: Period 1, Period 2, 24h

client.query(f"""
  UPDATE `{COHORT_TABLE}` p
  SET
    -- Period 1 slopes (IMV→eligible)
    fluid_intake_imv_to_eligible_slope  =
      ROUND(fluid_intake_imv_to_eligible  / NULLIF(duration_imv_to_eligible, 0), 4),
    fluid_output_imv_to_eligible_slope  =
      ROUND(fluid_output_imv_to_eligible  / NULLIF(duration_imv_to_eligible, 0), 4),
    fluid_balance_imv_to_eligible_slope =
      ROUND(fluid_balance_imv_to_eligible / NULLIF(duration_imv_to_eligible, 0), 4),
    -- Period 2 slopes (eligible→weaning)
    fluid_intake_eligible_to_weaning_slope  =
      ROUND(fluid_intake_eligible_to_weaning  / NULLIF(duration_eligible_to_weaning, 0), 4),
    fluid_output_eligible_to_weaning_slope  =
      ROUND(fluid_output_eligible_to_weaning  / NULLIF(duration_eligible_to_weaning, 0), 4),
    fluid_balance_eligible_to_weaning_slope =
      ROUND(fluid_balance_eligible_to_weaning / NULLIF(duration_eligible_to_weaning, 0), 4)
  WHERE cohort_criteria = 1
  AND   duration_imv_to_eligible > 0
""", job_config=job_config).result()
print('✓ Fluid balance slopes calculated')

In [ ]:
# ✅ Final fluid balance sanity check — all three windows
fluid_summary = client.query(f"""
  SELECT
    -- Period 1
    ROUND(AVG(fluid_balance_imv_to_eligible), 0)            AS mean_balance_p1_ml,
    -- Period 2
    ROUND(AVG(fluid_balance_eligible_to_weaning), 0)         AS mean_balance_p2_ml,
    -- 24h prior
    ROUND(AVG(fluid_balance_24h_prior), 0)                   AS mean_balance_24h_ml,
    -- Slope (the signal we actually use)
    ROUND(AVG(fluid_balance_eligible_to_weaning_slope), 3)   AS mean_balance_slope_mlph,
    COUNT(*)                                                 AS n
  FROM `{COHORT_TABLE}`
  WHERE cohort_criteria = 1
""", job_config=job_config).to_dataframe()

print('✅ Fluid balance summary — three windows:')
print(fluid_summary.to_string(index=False))
print()
print('Interpretation guide:')
print('  mean_balance_p1_ml:   Period 1 (IMV start→eligible) — expected large positive (days of fluids)')
print('  mean_balance_p2_ml:   Period 2 (eligible→weaning)   — smaller, closer to zero (diuresing phase)')
print('  mean_balance_24h_ml:  24h prior to extubation        — clinical team targeting balance')
print('  mean_balance_slope_mlph: Rate of change mL/h — negative = diuresing (good sign)')

# Visualise the slope distribution
slope_data = client.query(f"""
  SELECT fluid_balance_eligible_to_weaning_slope AS slope
  FROM `{COHORT_TABLE}`
  WHERE cohort_criteria = 1 AND fluid_balance_eligible_to_weaning_slope IS NOT NULL
  AND fluid_balance_eligible_to_weaning_slope BETWEEN -500 AND 500  -- exclude extreme outliers
""", job_config=job_config).to_dataframe()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(slope_data['slope'], bins=60, color='#3498db', alpha=0.7, edgecolor='white')
ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='Zero (balanced)')
ax.axvline(x=slope_data['slope'].mean(), color='green', linestyle='-', linewidth=1.5,
           label=f'Mean: {slope_data["slope"].mean():.1f} mL/h')
ax.set_xlabel('Fluid balance slope (mL/hour)', fontsize=11)
ax.set_ylabel('Number of patients', fontsize=11)
ax.set_title('Distribution of fluid balance slope\n(eligible → weaning window)', fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Negative slope = net diuresing — expected pattern for successfully weaned patients')

---
## Section 6: Outcomes — What Actually Happened?

### 🩺 Clinical Thinking

Defining 'successful extubation' seems simple. In practice, it requires explicit decisions:

| Outcome | Definition | Clinical Rationale |
|---------|------------|--------------------|
| **Success** | Alive AND not reintubated at 48h post-weaning | 48h is the clinical standard window for extubation failure — most failures occur within this period |
| **Failed — Reintubation** | Return to IMV within 48h | Second IMV episode detectable from measurement gap logic |
| **Failed — Death** | Death within 48h of weaning | Captured from death table |

### The Death-on-Ventilator Problem

This is a real clinical-data edge case worth understanding:

> *A patient deteriorates and the team decides to withdraw life support. The ventilator is switched off at 14:00. Death is declared and documented at 14:23. In the database: `first_imv_end = 14:00`, `died_ts = 14:23`. The gap is 23 minutes.*

Is this a weaning event? Clinically, no — it was planned withdrawal. But the data structure looks identical to a failed extubation followed by death.

The current approach excludes patients where death occurred within 12 hours of IMV end from the weaning cohort (handled in Section 3). This is a pragmatic approximation — in a future version, this should be refined using explicit withdrawal-of-care documentation if available.

**The lesson:** Outcome definitions require clinical judgment, not just code. Always document your assumptions.

In [ ]:
# Finalise cohort criteria and classify outcomes

# IMV duration
client.query(f"""
  UPDATE `{COHORT_TABLE}`
  SET first_imv_duration = TIMESTAMP_DIFF(first_weaning, first_imv_start, HOUR)
  WHERE first_imv_start IS NOT NULL""", job_config=job_config).result()

# cohort_criteria = 1
client.query(f"""
  UPDATE `{COHORT_TABLE}`
  SET cohort_criteria = 1
  WHERE first_weaning IS NOT NULL
  AND   first_imv_duration >= 24
  AND   admission_has_neuro = 0""", job_config=job_config).result()

# Success: alive and not reintubated at 48h
client.query(f"""
  UPDATE `{COHORT_TABLE}`
  SET weaning_success = 1, weaning_outcome = 'success'
  WHERE cohort_criteria = 1
  AND (weaning_to_second_imv IS NULL  OR weaning_to_second_imv >= 48)
  AND (died_ts IS NULL OR TIMESTAMP_DIFF(died_ts, first_weaning, HOUR) >= 48)""",
  job_config=job_config).result()

# Failure: reintubated OR died within 48h
client.query(f"""
  UPDATE `{COHORT_TABLE}`
  SET
    weaning_failure = 1,
    weaning_outcome = 'failure',
    weaning_outcome_detail = CASE
      WHEN weaning_to_second_imv IS NOT NULL AND weaning_to_second_imv < 48 THEN 'reintubated'
      WHEN died_ts IS NOT NULL AND TIMESTAMP_DIFF(died_ts, first_weaning, HOUR) < 48  THEN 'died'
      ELSE 'unknown'
    END
  WHERE cohort_criteria = 1
  AND (
    (weaning_to_second_imv IS NOT NULL AND weaning_to_second_imv < 48)
    OR (died_ts IS NOT NULL AND TIMESTAMP_DIFF(died_ts, first_weaning, HOUR) < 48)
  )""", job_config=job_config).result()

print('✓ Outcomes classified')

In [ ]:
# ✅ Sanity check: outcome distribution
outcomes = client.query(f"""
  SELECT
    COALESCE(weaning_outcome, 'unclassified') AS outcome,
    COALESCE(weaning_outcome_detail, '')       AS detail,
    COUNT(*)                                   AS n,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
  FROM `{COHORT_TABLE}`
  WHERE cohort_criteria = 1
  GROUP BY outcome, detail
  ORDER BY n DESC""", job_config=job_config).to_dataframe()

print('✅ Outcome distribution:')
print(outcomes.to_string(index=False))
print('\nExpect: ~70-80% success, ~15-25% failure')
print('If >50% failure: check your weaning definition — may be capturing too many brief extubations')
print('If >95% success: check your reintubation detection — may be missing failures')

# Visualise
fig, ax = plt.subplots(figsize=(8, 5))
colours = ['#2ecc71' if o == 'success' else '#e74c3c' for o in outcomes['outcome']]
bars = ax.bar(outcomes['outcome'] + '\n' + outcomes['detail'], outcomes['n'], color=colours)
for bar, row in zip(bars, outcomes.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{row.n:,}\n({row.pct}%)', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Number of patients')
ax.set_title('Weaning outcomes')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## What You've Built

```
patient_weaning table
│
├── Admission (visit_occurrence, 2010 + 2017, no 2002)
├── IMV episode (start, end, duration, 6h gap tolerance)
├── Neuro flag (condition_occurrence)
│
├── ELIGIBILITY TIMESTAMP ←─ First moment all criteria met
│     FiO2 ≤50%, PEEP ≤10
│     No norad, no propofol, no NMRA
│     ≥6h after admission
│
├── WEANING TIMESTAMP ←─── IMV end
│
├── Features at BOTH timepoints (14 variables × 2 = 28 columns)
│     Ventilator: FiO2, PEEP
│     Haemodynamics: SBP, DBP, SpO2
│     Haematology: Hb, WBC, platelets, HCT
│     Biochemistry: BE, CRP, creatinine, albumin, lactate
│
├── Slopes (trajectory per hour, 14 variables)
│     = (value_at_weaning - value_at_eligible) / hours
│
├── Fluid balance (3 periods × intake + output = 6 columns + derived)
│     IMV→eligible, eligible→weaning, 24h prior
│
└── Outcome: success / failure (reintubated / died)
```

### Reflecting on the Decisions

Every decision above maps to a clinical concept:

- **Excluding 2002** → Data quality is a form of study validity
- **Excluding neuro patients** → Clinical question specificity
- **Drug exclusions from eligibility** → Clinician intent matters, not just measurements
- **Two timestamps** → Trajectory requires at least two points
- **Slope with 4h minimum** → Clinical change takes time to manifest
- **Curated fluid IDs** → Methodology is in the detail

---

### What's Next

| Module | Content |
|--------|--------|
| **Module 3** | The BigTable pattern — how to build and maintain a cohort table incrementally across a team |

---
*ESICM LIVES26 Educational Series — Datathon Track*  
*Pipeline developed during 2026 ESICM INDICATE datathon*